---
# Phase 5 — Explore & Analysis

Goal: turn sentiment labels into actual product insight. This is where you find out *what* is driving negativity, not just *how much*.

**Input:** `indobert_scored_reviews.csv` from Phase 4
**Output:** a few summary CSVs + charts you can pull straight into a report

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("indobert_scored_reviews.csv")
df["at"] = pd.to_datetime(df["at"], errors="coerce")
df.shape

## 1. Final sentiment picture: star vs lexicon vs IndoBERT
Side by side, so you can see how much each method's story differs.

In [ ]:
compare = pd.DataFrame({
    "star_label": df["star_label"].value_counts(normalize=True),
    "lexicon_label": df["lexicon_label"].value_counts(normalize=True),
    "indobert_label": df["indobert_label"].value_counts(normalize=True),
}).fillna(0)

compare = compare.reindex(["positive", "neutral", "negative"])
compare.plot(kind="bar", figsize=(8, 5))
plt.ylabel("Proportion of reviews")
plt.title("Sentiment distribution: star rating vs lexicon vs IndoBERT")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

compare

## 2. Sentiment trend over time

Using IndoBERT's label since it's the most reliable of the three. A spike in negative sentiment is worth checking against release notes / known incidents.

In [ ]:
monthly = (
    df.dropna(subset=["at"])
      .set_index("at")
      .groupby([pd.Grouper(freq="ME"), "indobert_label"])
      .size()
      .unstack(fill_value=0)
)

monthly_pct = monthly.div(monthly.sum(axis=1), axis=0)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

monthly.plot(ax=axes[0])
axes[0].set_title("Review volume by sentiment per month")
axes[0].set_ylabel("Count")

monthly_pct.plot(ax=axes[1])
axes[1].set_title("Sentiment share per month")
axes[1].set_ylabel("Proportion")

plt.tight_layout()
plt.show()

## 3. Sentiment by app version

Which version had the worst sentiment share? Only versions with a reasonable review count are shown (low-volume versions produce noisy percentages).

In [ ]:
MIN_REVIEWS_PER_VERSION = 10

version_counts = df["app_version"].value_counts()
valid_versions = version_counts[version_counts >= MIN_REVIEWS_PER_VERSION].index

by_version = (
    df[df["app_version"].isin(valid_versions)]
    .groupby(["app_version", "indobert_label"])
    .size()
    .unstack(fill_value=0)
)
by_version_pct = by_version.div(by_version.sum(axis=1), axis=0).sort_values("negative", ascending=False)

by_version_pct.plot(kind="barh", stacked=True, figsize=(10, max(4, len(by_version_pct) * 0.4)))
plt.xlabel("Proportion")
plt.title("Sentiment share by app version (sorted by % negative)")
plt.tight_layout()
plt.show()

by_version_pct

## 4. Top keywords / phrases per sentiment class

Using the lightly-cleaned, unstemmed text — bigrams often carry more meaning than single words ("live chat", "tidak bisa", "customer service").

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def top_ngrams(texts, n=20, ngram_range=(1, 2)):
    vec = CountVectorizer(ngram_range=ngram_range, min_df=3)
    X = vec.fit_transform(texts.astype(str))
    counts = X.sum(axis=0).A1
    freq = pd.Series(counts, index=vec.get_feature_names_out()).sort_values(ascending=False)
    return freq.head(n)

negative_texts = df.loc[df["indobert_label"] == "negative", "clean_unstemmed"]
positive_texts = df.loc[df["indobert_label"] == "positive", "clean_unstemmed"]

print("=== Top terms in NEGATIVE reviews ===")
print(top_ngrams(negative_texts))

In [ ]:
print("=== Top terms in POSITIVE reviews ===")
print(top_ngrams(positive_texts))

## 5. Word cloud (negative reviews)

Quick visual gut-check — good for a slide/report, not for real analysis (a word cloud has no notion of statistical significance, just raw frequency).

In [ ]:
from wordcloud import WordCloud

text_blob = " ".join(negative_texts)
wc = WordCloud(width=1000, height=500, background_color="white", collocations=True).generate(text_blob)

plt.figure(figsize=(12, 6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Most frequent words — negative reviews")
plt.show()

## 6. Aspect tagging — what are people actually complaining about?

This is the most actionable part of this whole notebook. Instead of one big "negative" bucket, we tag each review against known PLN Mobile complaint categories using keyword rules. A review can match more than one aspect (e.g. a review about failed token purchase AND poor CS response).

**Extend `aspect_keywords` as you read more of your negative reviews** — this list is a starting point based on common complaint patterns, not exhaustive.

In [ ]:
aspect_keywords = {
    "login_otp": ["login", "otp", "masuk", "verifikasi", "daftar", "password", "kata sandi", "akun"],
    "token_pembelian": ["token", "beli", "pembelian", "kwh", "meter"],
    "pembayaran": ["bayar", "pembayaran", "transaksi", "saldo", "refund", "tagihan", "va bank"],
    "notifikasi_pemadaman": ["notif", "notifikasi", "pemberitahuan", "padam", "pemadaman", "listrik mati"],
    "crash_error": ["error", "bug", "crash", "lemot", "lambat", "force close", "keluar sendiri", "tidak bisa dibuka", "loading"],
    "customer_service": ["cs ", "customer service", "admin", "live chat", "respon", "layanan", "call center", "123"],
    "fitur_request": ["fitur", "tambahkan", "seharusnya ada", "belum ada", "semoga ada"],
    "poin_promo": ["poin", "reward", "hadiah", "promo", "diskon", "cashback"],
}

def tag_aspects(text, aspect_keywords=aspect_keywords):
    text = " " + str(text).lower() + " "
    tags = [aspect for aspect, kws in aspect_keywords.items() if any(kw in text for kw in kws)]
    return tags if tags else ["lainnya"]

negative_df = df[df["indobert_label"] == "negative"].copy()
negative_df["aspects"] = negative_df["clean_unstemmed"].apply(tag_aspects)

# explode so each aspect gets its own row for counting
aspect_counts = negative_df.explode("aspects")["aspects"].value_counts()
aspect_pct = (aspect_counts / len(negative_df) * 100).round(1)

print(f"Total negative reviews: {len(negative_df)}")
print("\n% of negative reviews mentioning each aspect (can exceed 100% since reviews can match multiple aspects):")
aspect_pct

In [ ]:
aspect_pct.sort_values().plot(kind="barh", figsize=(8, 5))
plt.xlabel("% of negative reviews")
plt.title("What negative reviews are actually about")
plt.tight_layout()
plt.show()

## 7. Sample reviews per aspect
Read a few from each category to sanity-check the tagging and pull real quotes for a report.

In [ ]:
ASPECT_TO_INSPECT = "customer_service"  # change this to explore other aspects

matching = negative_df[negative_df["aspects"].apply(lambda tags: ASPECT_TO_INSPECT in tags)]
print(f"{len(matching)} negative reviews tagged '{ASPECT_TO_INSPECT}'")

for _, row in matching.sample(min(5, len(matching)), random_state=1).iterrows():
    print(f"[{row['score']}★] {row['content']}")
    print("-" * 80)

## 8. Aspect trend over time
Is a particular complaint category getting worse recently?

In [ ]:
aspect_monthly = (
    negative_df.dropna(subset=["at"])
    .explode("aspects")
    .set_index("at")
    .groupby([pd.Grouper(freq="ME"), "aspects"])
    .size()
    .unstack(fill_value=0)
)

aspect_monthly.plot(figsize=(12, 6))
plt.title("Negative review aspects over time")
plt.ylabel("Count")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 9. Save summary outputs
These are the tables you'd actually put in a report.

In [ ]:
aspect_pct.to_csv("summary_aspect_breakdown.csv", header=["pct_of_negative_reviews"])
by_version_pct.to_csv("summary_sentiment_by_version.csv")
monthly_pct.to_csv("summary_sentiment_by_month.csv")
negative_df[["review_id", "content", "score", "at", "app_version", "aspects"]].to_csv("negative_reviews_tagged.csv", index=False)

print("Saved: summary_aspect_breakdown.csv, summary_sentiment_by_version.csv, summary_sentiment_by_month.csv, negative_reviews_tagged.csv")